In [ ]:
# Add your Kaggle credentials here before running
# Follow: https://www.kaggle.com/docs/api
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_creds = {
    "username": "YOUR_KAGGLE_USERNAME",
    "key": "YOUR_KAGGLE_API_KEY"
}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials configured")

Kaggle credentials configured


In [ ]:
!kaggle datasets list -s cicids2017

ref                                                      title                                              size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------  ------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
ericanacletoribeiro/cicids2017-cleaned-and-preprocessed  CICIDS2017: Cleaned & Preprocessed            210143955  2025-01-12 12:02:31.110000           8058         55                1  
ernie55ernie/improved-cicids2017-and-csecicids2018       Improved CICIDS2017 and CSECICIDS2018       10985642855  2023-08-15 16:57:44.847000           2198         21        0.8235294  
devendra416/ddos-datasets                                DDoS Dataset                                 2880543869  2019-04-30 13:49:22.867000          13719        152        0.7058824  
mdalamintalukder/cicids2017                              CICIDS2017   

In [ ]:
import kagglehub

path = kagglehub.dataset_download("ericanacletoribeiro/cicids2017-cleaned-and-preprocessed")
print("Dataset downloaded to:", path)

import os
for root, dirs, files_list in os.walk(path):
    for f in files_list:
        print(os.path.join(root, f))

100%|██████████| 200M/200M [00:01<00:00, 144MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/versions/6
/root/.cache/kagglehub/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/versions/6/cicids2017_cleaned.csv


In [ ]:
import pandas as pd

df = pd.read_csv('/root/.cache/kagglehub/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/versions/6/cicids2017_cleaned.csv')

print(df.shape)
print(df.columns.tolist())
df.head()

(2520751, 53)
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Max', 'Active Min', 'Idle Mean', '

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [ ]:
print(df['Attack Type'].value_counts())
print()
print(f"Total rows: {len(df)}")
print(f"Missing values per column:")
print(df.isnull().sum().sum(), "total nulls in dataset")

Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64

Total rows: 2520751
Missing values per column:
0 total nulls in dataset


In [ ]:
import pandas as pd

# Keep ALL attack samples (they're already scarce)
attack_types = ['DoS', 'DDoS', 'Port Scanning', 'Brute Force', 'Web Attacks', 'Bots']
df_attacks = df[df['Attack Type'].isin(attack_types)]

# Sample down Normal Traffic to a more reasonable size
# We'll keep 200,000 — still far more than any single attack type,
# but not 2 million-to-1,948 levels of imbalance
df_normal = df[df['Attack Type'] == 'Normal Traffic'].sample(n=200000, random_state=42)

df_balanced = pd.concat([df_normal, df_attacks], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(df_balanced['Attack Type'].value_counts())
print(f"\nTotal rows: {len(df_balanced)}")

Attack Type
Normal Traffic    200000
DoS               193745
DDoS              128014
Port Scanning      90694
Brute Force         9150
Web Attacks         2143
Bots                1948
Name: count, dtype: int64

Total rows: 625694


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
df_balanced['label_encoded'] = le.fit_transform(df_balanced['Attack Type'])

# Save this mapping — we'll need it to translate predictions back to readable labels
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_mapping)

X = df_balanced.drop(['Attack Type', 'label_encoded'], axis=1)
y = df_balanced['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining set:", X_train.shape)
print("Test set:", X_test.shape)

Label mapping: {'Bots': np.int64(0), 'Brute Force': np.int64(1), 'DDoS': np.int64(2), 'DoS': np.int64(3), 'Normal Traffic': np.int64(4), 'Port Scanning': np.int64(5), 'Web Attacks': np.int64(6)}

Training set: (500555, 52)
Test set: (125139, 52)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

start = time.time()

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

model.fit(X_train, y_train)

print(f"Training complete in {time.time() - start:.1f} seconds")

Training complete in 239.2 seconds


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import numpy as np

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Overall Accuracy: {accuracy:.4f}\n")

# Use the label encoder's classes in the correct order for readable output
target_names = le.classes_
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=target_names))

Overall Accuracy: 0.9988

Classification Report:
                precision    recall  f1-score   support

          Bots       0.94      0.96      0.95       389
   Brute Force       1.00      1.00      1.00      1830
          DDoS       1.00      1.00      1.00     25603
           DoS       1.00      1.00      1.00     38749
Normal Traffic       1.00      1.00      1.00     40000
 Port Scanning       1.00      1.00      1.00     18139
   Web Attacks       0.99      0.97      0.98       429

      accuracy                           1.00    125139
     macro avg       0.99      0.99      0.99    125139
  weighted avg       1.00      1.00      1.00    125139



In [ ]:
# Check for near-duplicate rows in the dataset (a known CICIDS2017 critique)
duplicate_count = df_balanced.duplicated(subset=X.columns.tolist()).sum()
print(f"Exact duplicate feature rows: {duplicate_count} out of {len(df_balanced)}")
print(f"Percentage: {duplicate_count/len(df_balanced)*100:.2f}%")

Exact duplicate feature rows: 68 out of 625694
Percentage: 0.01%


In [ ]:
importances = model.feature_importances_
feature_names = X.columns
top_indices = np.argsort(importances)[::-1][:15]

print("Top 15 most important features:")
for idx in top_indices:
    print(f"{feature_names[idx]}: importance = {importances[idx]:.4f}")

Top 15 most important features:
Destination Port: importance = 0.1296
Init_Win_bytes_backward: importance = 0.0613
Fwd Packet Length Max: importance = 0.0465
Subflow Fwd Bytes: importance = 0.0457
Packet Length Mean: importance = 0.0394
Average Packet Size: importance = 0.0392
Bwd Packet Length Mean: importance = 0.0389
Total Length of Fwd Packets: importance = 0.0379
Bwd Header Length: importance = 0.0337
Fwd Packet Length Mean: importance = 0.0314
Max Packet Length: importance = 0.0310
Bwd Packet Length Std: importance = 0.0262
Bwd Packet Length Max: importance = 0.0252
Packet Length Variance: importance = 0.0230
Init_Win_bytes_forward: importance = 0.0224


In [ ]:
import joblib

joblib.dump(model, 'intrusion_model.pkl')
joblib.dump(list(X.columns), 'intrusion_feature_columns.pkl')
joblib.dump(label_mapping, 'intrusion_label_mapping.pkl')

from google.colab import files
files.download('intrusion_model.pkl')
files.download('intrusion_feature_columns.pkl')
files.download('intrusion_label_mapping.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

# Save a sample of test rows (features + true label) for replay later
replay_sample = X_test.copy()
replay_sample['true_label'] = y_test.values
replay_sample['true_label_name'] = replay_sample['true_label'].map({v: k for k, v in label_mapping.items()})

# Shuffle and take a manageable sample for the demo (e.g. 500 rows)
replay_sample = replay_sample.sample(n=500, random_state=123).reset_index(drop=True)

replay_sample.to_csv('intrusion_replay_sample.csv', index=False)

from google.colab import files
files.download('intrusion_replay_sample.csv')

print(replay_sample['true_label_name'].value_counts())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

true_label_name
DoS               171
Normal Traffic    161
DDoS               88
Port Scanning      66
Brute Force         9
Web Attacks         3
Bots                2
Name: count, dtype: int64
